
# 41 复杂策略设计：多信号、状态机与执行

> **本章要解决的问题**：一个策略有三个信号、两种市场状态、一道回撤熔断和一层成本模型，它们怎么拼成一个能上账户的系统？为什么每个部件单独看都对，合起来回测很美、实盘很糟？

## 本章导读

第39章讲多因子合成，那是“横截面选品种”的视角；这一章换成“一个账户怎么跑一个策略”的视角：**信号、状态、仓位、订单、风控**五本账，任何一本记错，回测和实盘都会分叉。

按拼装顺序展开：多信号合成 → 状态机 → 入场/出场/止损 → 仓位 → 组合约束/执行/容量 → kill switch 与回测陷阱。

### 学习目标

把这一章当成一份上账户前的工作清单：

- **任务 1**：把 3 个信号合成一个目标仓位，写清权重口径与冲突处理。
- **任务 2**：用状态机写出状态、事件、转移、守卫，并说明最短停留的作用。
- **任务 3**：为同一策略写出 3 种出场规则，并解释止损为何降胜率却可能改善回撤。
- **任务 4**：用分数凯利和风险预算各算一次仓位。
- **任务 5**：列出组合约束和容量测算（参与率、冲击成本）。
- **任务 6**：给策略配一套 kill switch，背出至少 6 个回测陷阱。

### 先修

第02章期望/方差；第07章约束优化；第22章订单容量；第35章凯利公式；第39章多因子合成。

---

## 在量化研究中的位置

- **谁用**：策略研究员、组合经理、交易/执行、风控。
- **输入**：原始信号、行情、成本模型、账户与约束、风控限额。
- **输出**：目标仓位序列、订单计划、状态机日志、熔断记录、回测报告。
- **在流程的哪一步**：从信号到账户执行的最后一道工序；前面是 39，后面接 31 与 14。
- **失败模式**：信号重复暴露、状态机样本内调参、止损被扫、仓位超容量、熔断只写在文档里。
- **最小工作流**：信号 → 合成 → 状态门控 → 出入场 → 仓位 → 约束/执行 → 回测 → 熔断/监控。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="41 复杂策略设计：多信号、状态机与执行" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>41 复杂策略设计：多信号、状态机与执行｜知识图</title>
<desc>本章主线：信号只决定“想做什么”，状态机决定“允许做什么”，kill switch 最后兜底。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a41" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao41" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag41" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">41</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">复杂策略设计：多信号、状态机与执行</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：信号只决定“想做什么”，状态机决定“允许做什么”，kill switch 最后兜底。</text>
<rect x="65" y="185" width="125" height="62" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="127.5" y="208.8" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">空仓 / 观察</text>
<text x="127.5" y="227.3" font-size="8.8" fill="#6E7B8B" text-anchor="middle">等信号与风控</text>
<rect x="265" y="95" width="130" height="62" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="330" y="118.8" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">信号确认</text>
<text x="330" y="137.3" font-size="8.8" fill="#6E7B8B" text-anchor="middle">多信号投票</text>
<rect x="505" y="95" width="130" height="62" fill="#1B3A5C" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="570" y="118.8" font-size="12.5" fill="#FFFFFF" text-anchor="middle" font-weight="bold">持仓</text>
<text x="570" y="137.3" font-size="8.8" fill="#FFFFFF" text-anchor="middle">仓位与止损</text>
<rect x="505" y="280" width="130" height="62" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="570" y="303.9" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">退出 / 冷却</text>
<text x="570" y="322.3" font-size="8.8" fill="#6E7B8B" text-anchor="middle">平仓后不动</text>
<rect x="665" y="186" width="95" height="56" fill="#FFFFFF" stroke="#E07B39" stroke-width="1.5" rx="14"/>
<text x="712.5" y="206.5" font-size="10.5" fill="#E07B39" text-anchor="middle" font-weight="bold">Kill Switch</text>
<text x="712.5" y="224.1" font-size="8.2" fill="#6E7B8B" text-anchor="middle">一键平仓</text>
<path d="M 190 216 L 265 126" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a41)"/>
<rect x="204.5" y="157" width="46" height="14" fill="#FFFFFF"/>
<text x="227.5" y="166" font-size="10" fill="#6E7B8B" text-anchor="middle">信号触发</text>
<path d="M 395 126 L 505 126" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a41)"/>
<rect x="427" y="112" width="46" height="14" fill="#FFFFFF"/>
<text x="450" y="121" font-size="10" fill="#6E7B8B" text-anchor="middle">风控通过</text>
<path d="M 570 157 L 570 280" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a41)"/>
<rect x="538.8" y="204.5" width="62.5" height="14" fill="#FFFFFF"/>
<text x="570" y="213.5" font-size="10" fill="#6E7B8B" text-anchor="middle">止损 / 目标</text>
<path d="M 505 311 L 190 216" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a41)"/>
<rect x="324.5" y="249.5" width="46" height="14" fill="#FFFFFF"/>
<text x="347.5" y="258.5" font-size="10" fill="#6E7B8B" text-anchor="middle">冷却结束</text>
<path d="M 330 157 Q 240 205 150 185" fill="none" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="4 3" marker-end="url(#ag41)"/>
<rect x="223" y="197" width="46" height="14" fill="#FFFFFF"/>
<text x="246" y="206" font-size="10" fill="#6E7B8B" text-anchor="middle">信号消失</text>
<path d="M 665 210 Q 652 165 635 128" fill="none" stroke="#E07B39" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="5 3" marker-end="url(#ao41)"/>
<rect x="645" y="149" width="46" height="14" fill="#FFFFFF"/>
<text x="668" y="158" font-size="10" fill="#6E7B8B" text-anchor="middle">一键平仓</text>
<text x="65" y="372" font-size="9.6" fill="#6E7B8B">状态机规定“什么时候不允许交易”；kill switch 绕过策略逻辑，直接平仓</text>
<text x="65" y="390" font-size="9.6" fill="#1B3A5C">回测里也要有状态：否则同一信号会被重复加仓</text>
</svg>

> **图**：本章主线：信号只决定“想做什么”，状态机决定“允许做什么”，kill switch 最后兜底。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



---

## 1. 多信号合成：从一个信号到一个仓位

### 1.1 先统一口径：去极值、标准化、对齐时间

三个信号量纲可能完全不同：动量是收益率，carry 是价差，反转是成交额占比。直接相加等于让“数值大的信号”说话。合成前至少做三步：去极值、滚动标准化、对齐可交易时点。滚动标准化是

$$z_{i,t}=\frac{s_{i,t}-\operatorname{mean}_{60}(s_i)}{\operatorname{std}_{60}(s_i)}.$$

**公式拆解**
- **符号**：$s_{i,t}$ 是第 $i$ 个信号第 $t$ 期的原值，$\operatorname{mean}_{60}$、$\operatorname{std}_{60}$ 是过去 60 个交易日的滚动均值与标准差，$z_{i,t}$ 是标准化分数。
- **结构**：当前值相对近期均值的偏离除以近期波动，把不同量纲的信号换算成“偏离几个标准差”，且窗口只含过去、不含未来。
- **代入**：均值 0.02、标准差 0.05、今日值 0.065 时 $z=(0.065-0.02)/0.05=0.9$，另一个均值 25000、标准差 5000、今日 30000 的信号 $z$ 为 1.0。
- **边界**：窗口内标准差为 0 要返回缺失而不是无穷，极端值要先做去极值，用全样本均值方差就是未来函数。
- **用途**：多信号合成前统一口径、让不同量纲的信号可以相加；它不是信号强弱的绝对度量，窗口长度是平滑与滞后的权衡。

用滚动窗口而不是全样本，是因为全样本的均值和方差把未来信息搬到了现在——回测里叫未来函数。

### 1.2 三种权重口径

| 口径 | 权重 | 优点 | 风险 |
|---|---|---|---|
| 等权 | $1/N$ | 简单、稳 | 弱信号拖累强信号 |
| IC 加权 | $w_i\propto \mathrm{IC}_i$ | 偏向有效信号 | IC 用全样本估计=未来函数 |
| ICIR 加权 | $w_i\propto \mathrm{ICIR}_i$ | 兼顾有效性与稳定性 | ICIR 估计噪声大 |
| 风险预算 | $w_i\propto 1/\sigma_i$ | 控制风险贡献 | 与收益无关 |

**一条纪律**：权重只能用 $t$ 之前的数据估计。用全样本 IC 算权重是本章最高频的未来函数；实盘要用扩展窗口，接受权重会变、会抖。

### 1.3 冲突与净额：一多一空怎么办

1. **净额**：直接相加，由净信号决定方向。简单，但强弱会互相抵消；
2. **门控**：主导信号否决从属信号。保留方向性，但多了“谁主导”的参数；
3. **分账户**：各做各的，在组合层加总。便于归因，但成本和保证金翻倍。

无论选哪种，都要记录**信号同向率**：太低说明信号在打架，太高说明重复、加不出信息。

### 1.4 代码：三信号合成与权重口径对比



In [2]:

import numpy as np

rng = np.random.default_rng(4101)
T = 1500
names = ["趋势", "carry", "反转"]

common = rng.normal(0, 1, T)
raw = {}
for name in names:
    own = rng.normal(0, 1, T)
    raw[name] = 0.4 * common + np.sqrt(0.84) * own          # 信号之间相关 0.16

# 下一期收益由三个信号共同驱动：信号 t 决定收益 t+1
shock = 0.06 * raw["趋势"] + 0.04 * raw["carry"] + 0.02 * raw["反转"] + rng.normal(0, 1, T)
ret = np.r_[0.0, 0.0002 + 0.011 * shock[:-1] / shock.std()]


def rolling_z(x, win=60):
    out = np.full(len(x), np.nan)
    for t in range(win, len(x)):
        w = x[t - win:t]
        out[t] = (x[t] - w.mean()) / w.std(ddof=1)
    return out


Z = {k: rolling_z(v) for k, v in raw.items()}
idx = np.arange(80, T - 1)
y = ret[idx + 1]
Zt = {k: v[idx] for k, v in Z.items()}

print("信号两两相关：")
for i, a in enumerate(names):
    for b in names[i + 1:]:
        print(f"  {a}-{b}: {np.corrcoef(Zt[a], Zt[b])[0, 1]:.2f}")

ic_hat = {k: float(np.corrcoef(Zt[k], y)[0, 1]) for k in names}
print("全样本 IC（回测这样估权重就是未来函数）：", {k: round(v, 3) for k, v in ic_hat.items()})

w_eq = {k: 1 / 3 for k in names}
w_ic = {k: ic_hat[k] / sum(ic_hat.values()) for k in names}
inv = {k: 1 / Zt[k].std(ddof=1) for k in names}
w_rp = {k: inv[k] / sum(inv.values()) for k in names}

for label, w in [("等权", w_eq), ("IC加权", w_ic), ("风险预算", w_rp)]:
    comp = sum(w[k] * Zt[k] for k in names)
    pos = np.tanh(comp)
    pnl = pos * y
    sharpe = pnl.mean() / pnl.std(ddof=1) * np.sqrt(252)
    turnover = np.abs(np.diff(pos)).mean()
    agree = np.mean(np.sign(Zt["趋势"]) == np.sign(Zt["carry"]))
    print(f"{label}: 权重 {np.round([w[k] for k in names], 3)}, Sharpe {sharpe:.2f}, "
          f"日换手 {turnover:.3f}, 趋势/carry 同向率 {agree:.0%}")


信号两两相关：
  趋势-carry: 0.16
  趋势-反转: 0.14
  carry-反转: 0.11
全样本 IC（回测这样估权重就是未来函数）： {'趋势': 0.069, 'carry': 0.05, '反转': 0.01}
等权: 权重 [0.333 0.333 0.333], Sharpe 0.96, 日换手 0.588, 趋势/carry 同向率 56%
IC加权: 权重 [0.536 0.386 0.078], Sharpe 1.12, 日换手 0.621, 趋势/carry 同向率 56%
风险预算: 权重 [0.334 0.333 0.333], Sharpe 0.96, 日换手 0.588, 趋势/carry 同向率 56%



**读数与应用**：信号两两相关 0.1~0.2，合成收益不是简单相加。IC 加权把权重压向样本内最有效的趋势信号，Sharpe 从 0.96 抬到 1.12；风险预算在标准化后几乎等于等权——z-score 已统一波动，所以风险预算≈等权。注意 IC 是全样本估的，真实可交易版本会更差。

**常见错误**：

1. 不标准化直接相加，结果由量纲最大的信号决定。
2. 用全样本 IC 估权重，回测虚高。
3. 只看合成 Sharpe，不看同向率：太高说明信息重复，太低说明在打架。

**停下来想一想**：三个信号两两相关 0.6，等权合成后信息量等于 3 个还是接近 1 个？

<details><summary>对照你的思路</summary>

接近 1 个。相关性越高，合成信号越像“同一个信号的不同说法”，权重优化挤出的增量越少。看相关矩阵和合成前后的 IC/Sharpe 就能判断；这也是第39章讲正交化的原因。

</details>

---

## 2. Regime 与状态机：什么时候信哪个信号

### 2.1 regime 的三种定义与用途

- **趋势/震荡**：均线斜率、ADX、20 日收益的 z 值；
- **高波/低波**：已实现波动率的历史分位；
- **危机/正常**：相关性、流动性、成交/持仓异常。

regime 不是预测器，而是**开关**：它不告诉你明天涨跌，只告诉你“这套参数现在该不该用”。趋势策略在震荡市被反复止损，均值回归在趋势市逆势加仓——regime 用来避免用错工具。

### 2.2 状态机的五要素

| 要素 | 含义 | 例子 |
|---|---|---|
| 状态 S | 系统当前所处的位置 | 震荡、上行趋势、下行趋势 |
| 事件 E | 触发判断的观测 | 趋势分 $z$ 突破阈值 |
| 守卫 G | 转移的附加条件 | 最短停留 10 天、连续 3 日确认 |
| 转移 T | 从哪个状态到哪个状态 | 震荡 → 上行趋势 |
| 动作 A | 进入状态后做什么 | 允许趋势信号、仓位乘数 1.0 |

状态机的价值是**可审计**：事后能查“哪天因为什么切到哪个状态”，而不是一句“模型觉得该减仓”。

### 2.3 滞回与最短停留：为什么不能只用一个阈值

单阈值会抖：趋势分在 1.0 附近来回穿，状态一天一变，换手和成本爆炸。两个补丁：**滞回**（进 1.0、出 0.3，中间保持）和**最短停留**（每个状态至少 $N$ 天）。代价是反应变慢：趋势末端回吐更多、转折确认更晚。但状态机本来就不是抓拐点的，而是**不在错误的市场里用错误的策略**。

### 2.4 代码：趋势三态 + 波动两档的状态机



In [3]:

import numpy as np

rng = np.random.default_rng(4102)
T = 1200
seg = np.repeat([0, 1, 0, -1, 0, 1, -1, 0], 150)       # 教学价格：分段漂移
drift = np.select([seg == 1, seg == -1], [0.0025, -0.0025], 0.0)
close = 100 * np.cumprod(1 + drift + rng.normal(0, 0.008, T))
ret = np.r_[0.0, np.diff(close) / close[:-1]]

ret20 = np.array([close[t] / close[max(0, t - 20)] - 1 for t in range(T)])
vol20 = np.array([ret[max(0, t - 20):t + 1].std(ddof=1) if t >= 20 else 0.01
                  for t in range(T)])
trend_score = ret20 / (vol20 * np.sqrt(20) + 1e-12)

state = 0                                 # 0 震荡，1 上行，2 下行
states = np.zeros(T, dtype=int)
dwell = 10
for t in range(20, T):
    dwell += 1
    z = trend_score[t]
    if state == 0 and abs(z) > 1.0 and dwell >= 10:
        state = 1 if z > 0 else 2
        dwell = 0
    elif state != 0 and abs(z) < 0.3 and dwell >= 10:
        state = 0
        dwell = 0
    states[t] = state

vol_hi = np.array([vol20[t] > np.median(vol20[max(0, t - 250):t + 1]) for t in range(T)])
regime = states * 2 + vol_hi.astype(int)
labels = ["低波震荡", "高波震荡", "低波上行", "高波上行", "低波下行", "高波下行"]

fwd = np.r_[ret[1:], np.nan]
for i, lab in enumerate(labels):
    m = (regime == i) & ~np.isnan(fwd)
    print(f"{lab}: 占比 {m.mean():.0%}, 次日均值 {fwd[m].mean() * 1e4:+.1f}bp, "
          f"胜率 {(fwd[m] > 0).mean():.0%}")

pos = np.where(states == 1, 1.0, np.where(states == 2, -1.0, 0.0))
pnl = np.r_[0.0, pos[:-1]] * ret
print(f"状态切换 {int(np.sum(np.diff(states) != 0))} 次，趋势状态占比 {(states != 0).mean():.0%}，"
      f"只做趋势状态的年化 Sharpe {pnl.mean() / pnl.std(ddof=1) * np.sqrt(252):.2f}")


低波震荡: 占比 20%, 次日均值 -7.1bp, 胜率 44%
高波震荡: 占比 21%, 次日均值 -0.9bp, 胜率 50%
低波上行: 占比 15%, 次日均值 +14.3bp, 胜率 53%
高波上行: 占比 8%, 次日均值 +2.5bp, 胜率 55%
低波下行: 占比 18%, 次日均值 -6.5bp, 胜率 47%
高波下行: 占比 18%, 次日均值 -7.0bp, 胜率 49%
状态切换 55 次，趋势状态占比 59%，只做趋势状态的年化 Sharpe 1.24



**读数与应用**：低波上行状态次日均值 +14.3bp、胜率 53%；下行状态为负，做空有利；震荡状态接近 0。注意：状态用**当天收盘**算、仓位第二天生效，代码用 `pos[:-1] * ret[1:]` 对齐；状态机的价值是“避开错误的市场”，不是自己赚钱。

**常见错误**：

1. 状态用当天收盘算、仓位用当天收益算——未来函数。
2. 只报“状态准确率”，不报状态下的收益/风险。
3. 没有滞回与最短停留，换手高得离谱；状态越加越多，每档只剩几天样本。

**停下来想一想**：状态机“准确率”高，就一定赚钱吗？

<details><summary>对照你的思路</summary>

不一定。判对状态但切换太晚，或切换成本高于避开的损失，净收益仍可能为负。评价看切换频率、成本和被门控掉那些日子的收益。

</details>

---

## 3. 入场、出场与止损

### 3.1 入场：阈值、确认与时间窗

入场规则要写清三件事：**触发条件**（信号过阈值）、**确认方式**（连续 $N$ 日、或突破后回踩不破）、**时间窗**（避开开盘/收盘前 15 分钟与交割月）。阈值越低交易越多、成本越高；越高机会越少。确认能减少假突破，但会让入场价变差。

### 3.2 出场：四种典型规则

| 出场规则 | 逻辑 | 优点 | 代价 |
|---|---|---|---|
| 信号反转 | 信号穿越 0 | 让利润跑 | 回吐大 |
| 固定止损 | 跌破入场价 − $k\cdot\mathrm{ATR}$ | 控制单笔亏损 | 易被噪声扫掉 |
| 移动止损 | 跌破最高点 − $k\cdot\mathrm{ATR}$ | 锁定利润 | 回调时提前离场 |
| 时间止损 | 持仓超过 $N$ 天 | 释放占用 | 可能错过慢趋势 |

### 3.3 止损的代价：胜率下降换回撤改善

止损最反直觉的地方：它通常**降低胜率**——很多单子本来能扛回来，止损把它们变成实亏；但它剪掉了亏损的长尾，平均亏损变小、最大回撤可能改善。所以要看**期望、盈亏比、最大回撤**三个一起。

顺序上**先定每笔风险，再定止损距离，最后反推手数**。反过来“先手数后止损”，风险会随波动率漂移。

### 3.4 代码：同一批入场点，四种出场规则



In [4]:

import numpy as np

rng = np.random.default_rng(4103)
T = 1500
seg = np.repeat([0, 1, 0, -1, 0, 1, -1, 0], 188)[:T]   # 教学价格：分段趋势
drift = np.select([seg == 1, seg == -1], [0.0018, -0.0018], 0.0)
r = drift + rng.normal(0, 0.010, T)
close = 100 * np.cumprod(1 + r)
atr = np.array([close[t] * r[max(0, t - 14):t].std(ddof=1) if t >= 15 else close[t] * 0.01
                for t in range(T)])
sig = np.zeros(T)
for t in range(20, T):
    sig[t] = (close[t] / close[t - 20] - 1) / (r[t - 20:t].std(ddof=1) * np.sqrt(20))

entries = [t for t in range(21, T - 1) if sig[t] > 0.8 and sig[t - 1] <= 0.8]


def run(rule, stop_mult=2.5, hold_max=20, cost_bp=2.0, max_hold=250):
    trades = []
    for t0 in entries:
        entry, peak, exit_ret = close[t0], close[t0], None
        for t in range(t0 + 1, min(T - 1, t0 + max_hold)):
            peak = max(peak, close[t])
            if rule == "signal":
                hit = sig[t] < 0
            elif rule == "atr":
                hit = close[t] < entry - stop_mult * atr[t]
            elif rule == "trail":
                hit = close[t] < peak - stop_mult * atr[t]
            else:
                hit = t - t0 >= hold_max
            if hit:
                exit_ret = close[t] / entry - 1
                break
        if exit_ret is None:                        # 没触发任何出场：窗口末强制平仓
            exit_ret = close[t] / entry - 1
        trades.append(exit_ret - 2 * cost_bp / 1e4)
    tr = np.array(trades)
    eq = np.cumprod(1 + tr)
    aw = tr[tr > 0].mean() if (tr > 0).any() else float("nan")
    al = tr[tr <= 0].mean() if (tr <= 0).any() else float("nan")
    return (len(tr), (tr > 0).mean(), aw, al, tr.mean(),
            eq[-1] - 1, (eq / np.maximum.accumulate(eq) - 1).min())


print(f"{'出场规则':<8}{'笔数':>4}{'胜率':>7}{'均盈':>9}{'均亏':>9}{'期望':>9}{'累计':>9}{'回撤':>9}")
for rule, label in [("signal", "信号反转"), ("atr", "固定ATR"), ("trail", "移动ATR"), ("time", "时间止损")]:
    n, win, aw, al, exp, total, mdd = run(rule)
    print(f"{label:<8}{n:>4}{win:>7.0%}{aw:>9.2%}{al:>9.2%}{exp:>9.2%}{total:>9.1%}{mdd:>9.1%}")


出场规则      笔数     胜率       均盈       均亏       期望       累计       回撤
信号反转      44    23%    6.25%   -2.33%   -0.38%   -22.4%   -30.6%
固定ATR     44    16%   27.34%   -2.93%    1.89%    62.7%   -41.1%
移动ATR     44    34%    5.09%   -2.06%    0.38%     7.8%   -20.7%
时间止损      44    43%    2.64%   -3.81%   -1.03%   -38.5%   -46.1%



**读数与应用**：四种出场用同一批入场点，未触发时窗口末平仓。结果很说明问题：信号反转胜率 23%、期望为负，它要吃完整个回撤才离场；固定 ATR 止损胜率最低（16%），但盈亏比和累计最高、回撤也最大——只止损不止盈，就是“小亏很多次、偶尔大赚”；移动 ATR 最均衡；时间止损胜率最高、期望最差。

**常见错误**：

1. 用“胜率”排序出场规则——高胜率往往对应小赚大亏。
2. 止损写死百分比，不随波动率调整。
3. 先定手数再设止损，单笔风险随波动漂移。
4. 回测当天触发、当天收盘价成交，忽略下一根 K 线。

**停下来想一想**：止损从 2.5 倍 ATR 收紧到 1 倍，胜率通常会怎么变？期望呢？

<details><summary>对照你的思路</summary>

胜率通常下降——更多单子被扫掉；平均亏损变小，但交易次数和成本上升，期望不一定改善。极端时（噪声大、趋势弱）期望会由正转负。止损距离要和信号噪声匹配，不是越紧越安全。

</details>

---

## 4. 仓位：凯利、风险预算与波动率目标

### 4.1 凯利：数学最优，不是心理最优

离散下注的凯利比例是

$$f^*=\frac{bp-q}{b}=\frac{p(b+1)-1}{b},$$

其中 $p$ 是胜率、$q=1-p$、$b$ 是赔率（赢时赚本金的 $b$ 倍）；连续近似下 $f^*\approx\mu/\sigma^2$。

凯利有两个致命前提：**胜率和赔率已知且稳定**，**每笔独立同分布**。真实策略里 $p$、$b$ 都是估的，还有自相关和尾部，参数估高一点就会过度下注。所以实务常用**半凯利、1/4 凯利**：牺牲一点增长率，换回撤大幅下降。

### 4.2 风险预算：从“每笔风险”反推手数

先定单笔风险 $R$（如权益的 1%），再定止损距离 $d$（价格单位），合约乘数 $m$：

$$\text{手数}=\left\lfloor\frac{R\cdot E}{d\cdot m}\right\rfloor.$$

**公式拆解**
- **符号**：$R$ 是单笔风险占账户权益的比例，$E$ 是账户权益，$d$ 是入场价与止损价的距离，$m$ 是合约乘数，$\lfloor\cdot\rfloor$ 表示向下取整。
- **结构**：分子 $R\cdot E$ 是这笔交易愿意亏的钱，分母 $d\cdot m$ 是一手触发止损的亏损，两者相除得到手数，止损越宽手数越少。
- **代入**：$E=100$ 万元、$R=1\%$、$d=50$ 元/吨、$m=10$ 吨/手时风险预算 1 万元、每手亏 500 元，手数为 $\lfloor10000/500\rfloor=20$ 手。
- **边界**：$d\to0$ 时手数爆炸、必须设最小止损或上限，取整、跳空、滑点都会让实际亏损超过 $R$。
- **用途**：把“每笔只亏固定比例”的风险预算落到具体手数；它不是凯利，$R$ 是你选的，也不能保证不爆仓。

好处是止损宽自动减仓、止损窄自动加仓，每笔亏损钉在 $R$ 附近；代价是取整、跳空、滑点会让实际亏损偏离 $R$。

### 4.3 波动率目标：自动降杠杆，但有滞后

$$L=\min\left(L_{\max},\frac{\sigma_{\text{target}}}{\hat\sigma}\right).$$

**公式拆解**
- **符号**：$L$ 是目标杠杆倍数（1 为满仓），$\sigma_{\text{target}}$ 是目标年化波动率，$\hat\sigma$ 是当前估计的年化波动率，$L_{\max}$ 是杠杆或仓位上限。
- **结构**：用目标波动除以实际波动来缩放仓位，实际波动偏高就缩小、偏低就放大，外层 $\min$ 防止低波动时加出极端杠杆。
- **代入**：$\sigma_{\text{target}}=10\%$、$\hat\sigma=16\%$ 得 $L=0.625$、仓位打 62.5 折；若 $\hat\sigma=8\%$ 则 $L=1.25$，但上限为 1 时只能取 1。
- **边界**：$\hat\sigma\to0$ 时 $L$ 发散要靠 $L_{\max}$ 截断，$\hat\sigma$ 永远滞后：危机时波动率先跳升、系统才开始降仓，已经晚了半拍。
- **用途**：让风险预算保持稳定、避免“越赚越重、危机一把亏光”；它不是收益增强工具，只治长期风险漂移，治不了瞬时尾部。

波动率目标让风险预算保持稳定，但波动率估计永远滞后：危机爆发时 $\hat\sigma$ 上升，系统才开始降仓，已经晚了半拍。它治“长期风险漂移”，不治“瞬时尾部”。

### 4.4 代码：凯利、风险预算与蒙特卡洛



In [5]:

import numpy as np

p, b = 0.45, 1.5
kelly = (p * b - (1 - p)) / b
print(f"胜率 {p:.0%}、赔率 {b}: 全凯利 {kelly:.3f}，半凯利 {kelly / 2:.3f}，1/4 凯利 {kelly / 4:.3f}")

equity, risk_pct, stop_dist, mult = 1_000_000, 0.01, 60, 10
lots = int(equity * risk_pct / (stop_dist * mult))
print(f"风险预算：权益 {equity:,}、每笔风险 {risk_pct:.0%}、止损 {stop_dist} 元/吨、乘数 {mult}")
print(f"  → {lots} 手，实际每笔风险 {lots * stop_dist * mult:,.0f} 元"
      f"（{lots * stop_dist * mult / equity:.2%}）")

target_vol, realized_vol, lev_cap = 0.10, 0.20, 2.0
print(f"波动率目标：目标 {target_vol:.0%} / 当前 {realized_vol:.0%} → 杠杆 {min(lev_cap, target_vol / realized_vol):.2f}")


def sim(f, seed=4104, n_paths=500, n_bets=250):
    rng = np.random.default_rng(seed)
    win = rng.random((n_paths, n_bets)) < p
    nav = np.cumprod(1 + np.where(win, b * f, -f), axis=1)
    mdd = (nav / np.maximum.accumulate(nav, axis=1) - 1).min(axis=1)
    return (np.median(nav[:, -1]), np.percentile(nav[:, -1], 5),
            np.median(mdd), (nav.min(axis=1) < 0.5).mean())


for f, label in [(kelly, "全凯利"), (kelly / 2, "半凯利"), (kelly / 4, "1/4 凯利")]:
    med, p5, mdd, ruin = sim(f)
    print(f"{label} f={f:.3f}: 中位终值 {med:.2f}，5% 分位 {p5:.2f}，"
          f"中位最大回撤 {mdd:.0%}，净值腰斩概率 {ruin:.0%}")


胜率 45%、赔率 1.5: 全凯利 0.083，半凯利 0.042，1/4 凯利 0.021
风险预算：权益 1,000,000、每笔风险 1%、止损 60 元/吨、乘数 10
  → 16 手，实际每笔风险 9,600 元（0.96%）
波动率目标：目标 10% / 当前 20% → 杠杆 0.50
全凯利 f=0.083: 中位终值 3.27，5% 分位 0.28，中位最大回撤 -75%，净值腰斩概率 44%
半凯利 f=0.042: 中位终值 2.50，5% 分位 0.72，中位最大回撤 -46%，净值腰斩概率 11%
1/4 凯利 f=0.021: 中位终值 1.72，5% 分位 0.92，中位最大回撤 -26%，净值腰斩概率 0%



**读数与应用**：全凯利增长最快，但中位最大回撤 −75%、腰斩概率 44%；减半后中位终值只降约四分之一，回撤和腰斩概率大幅下降——这就是分数凯利的性价比。风险预算把“1% 风险”翻译成 16 手（实际 0.96%）；波动率目标把 20% 的当前波动压到 10%，杠杆 0.5 倍。

**常见错误**：

1. 用回测的胜率/赔率直接上全凯利，等于假设参数永远不变。
2. 先算手数再设止损，风险失控。
3. 波动率目标用“预测波动”却忘了估计误差。
4. 忽略保证金和涨跌停。

---

## 5. 组合约束、执行与容量

### 5.1 约束清单

| 约束 | 度量 | 常见处理 |
|---|---|---|
| 单品种 | 名义暴露、保证金 | 截断/上限 |
| 板块 | 同板块净暴露 | 净额后限额 |
| 杠杆 | 总名义/权益 | 等比缩放 |
| 集中度 | HHI、前五大占比 | 惩罚项 |
| 多空 | 净暴露、总暴露 | 目标区间 |
| 流动性 | ADV 参与率 | 拆单/限仓 |

约束不是越严越安全：太严策略没 alpha，太松一次极端行情就击穿风控。先写清业务理由，再放进优化器。

### 5.2 执行：回测和实盘分叉的地方

- **下单方式**：市价单快但贵；限价单便宜但可能不成交；TWAP/VWAP 按时间或成交量拆单。
- **冲击成本**：规模越大越贵，直观形式 $\text{cost}\approx c\,\sigma\sqrt{Q/V}$（系数是教学示例，要按品种实测）。
- **时滞**：信号 $t$ 收盘生成，最早 $t+1$ 开盘成交；用 $t$ 收盘价成交就是未来函数。
- **未成交**：限价单可能只成交一部分，回测要假设最坏情况。

### 5.3 容量：不是“能装多少钱”

容量 = 在**给定的冲击成本预算**下还能装多少资金。规模上升 → 参与率上升 → 冲击上升 → 净收益下降，容量曲线先平后陡。测法很朴素：把成本逐档提高，看净 Sharpe 在哪一档跌破可接受线。

### 5.4 代码：多策略组合 + 成本扫描 + 回撤熔断



In [6]:

import numpy as np

rng = np.random.default_rng(4105)
T = 2000
common = rng.normal(0, 1, T)                   # 共同暴露，让三策略相关
spec = [("趋势", 4.0e-4, 0.016, 0.25), ("carry", 2.5e-4, 0.012, 0.10), ("反转", 2.0e-4, 0.014, -0.15)]
R = np.column_stack([
    mu + sd * (beta * common + np.sqrt(1 - beta ** 2) * rng.normal(0, 1, T))
    for _, mu, sd, beta in spec
])
vol = R.std(axis=0, ddof=1) * np.sqrt(252)
w = (1 / vol) / (1 / vol).sum()
names = [s[0] for s in spec]
print("单策略年化波动", {n: round(float(v), 3) for n, v in zip(names, vol)})
print("风险预算权重", {n: round(float(v), 3) for n, v in zip(names, w)})

gross = R @ w
turnover = 0.08                                 # 教学假设：日均换手 8%
for c_bps in [1, 5, 10, 20]:
    net = gross - turnover * c_bps / 1e4
    print(f"成本 {c_bps:>2}bp: 年化 {net.mean() * 252:.2%}, "
          f"Sharpe {net.mean() / net.std(ddof=1) * np.sqrt(252):.2f}")

net = gross - turnover * 5 / 1e4
nav = np.cumprod(1 + net)
dd = nav / np.maximum.accumulate(nav) - 1

nav_ks, dd_ks, exposure = np.ones(T), np.zeros(T), np.ones(T)
peak = 1.0
for t in range(1, T):
    prev = dd_ks[t - 1]
    if prev < -0.15:                            # 熔断档位：回撤越深，仓位越低
        e = 0.25
    elif prev < -0.10:
        e = 0.5
    elif prev > -0.08:
        e = 1.0
    else:
        e = exposure[t - 1]
    exposure[t] = e
    nav_ks[t] = nav_ks[t - 1] * (1 + net[t] * e)
    peak = max(peak, nav_ks[t])
    dd_ks[t] = nav_ks[t] / peak - 1

print(f"5bp 成本、无熔断：年化 {net.mean() * 252:.2%}, 最大回撤 {dd.min():.2%}")
print(f"5bp 成本、有熔断：年化 {(net * exposure).mean() * 252:.2%}, 最大回撤 {dd_ks.min():.2%}, "
      f"平均仓位 {exposure.mean():.0%}, 降仓天数 {int((exposure < 1).sum())}")


单策略年化波动 {'趋势': 0.254, 'carry': 0.18, '反转': 0.22}
风险预算权重 {'趋势': 0.28, 'carry': 0.396, '反转': 0.324}
成本  1bp: 年化 12.51%, Sharpe 1.02
成本  5bp: 年化 11.71%, Sharpe 0.95
成本 10bp: 年化 10.70%, Sharpe 0.87
成本 20bp: 年化 8.68%, Sharpe 0.71
5bp 成本、无熔断：年化 11.71%, 最大回撤 -18.02%
5bp 成本、有熔断：年化 10.59%, 最大回撤 -15.11%, 平均仓位 86%, 降仓天数 564



**读数与应用**：成本从 1bp 抬到 20bp，净 Sharpe 从 1.02 掉到 0.71——这就是容量的方向。熔断把最大回撤从 −18.0% 压到 −15.1%，代价是年化从 11.7% 降到 10.6%：**熔断不是免费保险，是用收益换生存概率。** 若熔断后回撤反而更大，说明触发太晚或恢复太急。

**常见错误**：

1. 回测默认“无限流动性、零冲击”，容量算出来大得离谱。
2. 用收盘价成交，忽略 $t+1$ 的时滞。
3. 熔断只降仓位、不记录状态，重启后自动满仓复活。
4. 成本只算手续费，不算冲击和价差。

**补充**：日换手 80% 时双边成本约 16%/年，足以吃掉大部分超额收益。

---

## 6. Kill switch 与回测陷阱

### 6.1 kill switch 的六个触发器

| 触发器 | 阈值（教学示例） | 动作 |
|---|---|---|
| 单日亏损 | 权益 −2% | 降半仓，当日不再加仓 |
| 回撤 | −10% / −15% | 降半仓 / 降到 1/4 仓 |
| 连续亏损 | 8 笔 | 暂停开新仓，人工复核 |
| 数据异常 | 缺 bar、跳价、字段为空 | 冻结新开仓 |
| 执行异常 | 拒单率 > 20%、滑点 > 3σ | 暂停该品种 |
| 人工/合规 | 风控指令 | 全部平仓 |

阈值是教学示例；真实阈值按策略波动、资金性质和监管要求定。关键是触发器必须**写进代码、落盘、可审计**。

### 6.2 熔断不是“重启”

合格的 kill switch 要满足三条：**状态持久化**（重启后仍是熔断态）、**恢复人工确认**（不能自动满仓复活）、**有复盘记录**。它只降风险，不判断方向。

### 6.3 回测陷阱清单

1. **未来函数**：$t$ 收盘信号、$t$ 收盘价成交；全样本 IC 估权重。
2. **幸存者偏差**：只用今天还活着的品种。
3. **状态机参数样本内调**：阈值、最短停留都是拟合的。
4. **止损参数过拟合**：参数是“看着历史选的”。
5. **忽略成本/冲击**：换手越高越致命。
6. **忽略时滞与未成交**：信号→下单→成交至少差一个 bar。
7. **多重检验**：试了 50 组参数，只报最好那组（见第20章）。
8. **容量幻觉**：回测能买无限量，实盘一买就涨。

### 6.4 上账户前的自查清单

- [ ] 信号、权重、状态机参数只用历史数据估计；
- [ ] 信号与成交至少留一个 bar 时滞；
- [ ] 成本含手续费、价差、冲击；
- [ ] 仓位有单笔风险、品种、杠杆上限；
- [ ] 状态机日志与 kill switch 都落盘；
- [ ] 样本外验证、容量测算都做过。

---

## 本章小结

### 一张表

| 部件 | 输入 | 输出 | 典型失败 |
|---|---|---|---|
| 信号合成 | 多个原始信号 | 目标仓位 | 权重用全样本、信号重复 |
| 状态机 | 趋势/波动指标 | 门控开关 | 抖动、样本内调参 |
| 出入场 | 信号与价格 | 交易序列 | 止损被扫、时滞 |
| 仓位 | 风险预算/凯利 | 手数 | 先手数后止损 |
| 约束/执行 | 账户与成本 | 可执行订单 | 忽略冲击与容量 |
| kill switch | 损益与异常 | 降仓/停机 | 不落盘、自动复活 |

### 八句话

1. 合成前先标准化，否则量纲最大的信号说了算。
2. 权重只能用历史估，全样本 IC 是未来函数。
3. 状态机不是预测器，是开关；滞回和最短停留是它的保险。
4. 止损通常降低胜率，但可能换来更小的回撤。
5. 先定每笔风险，再定止损，最后反推手数。
6. 全凯利在参数误差下很危险，分数凯利更实用。
7. 成本扫描就是容量测试：净 Sharpe 跌破可接受线就是规模上限。
8. 熔断是生存工具，不是收益工具。

---

## 本章习题

**1.** 合成前为什么要滚动 z-score？用全样本 z-score 有什么问题？

**2.** 信号 A 的 IC 是 0.06、B 是 0.02，直接 IC 加权有什么风险？

**3.** 状态机的“滞回”和“最短停留”各解决什么？代价是什么？

**4.** 止损从 2.5 倍 ATR 收紧到 1 倍，胜率、交易次数、期望可能怎么变？

**5.** 胜率 55%、赔率 1.2，全凯利比例多少？半凯利多少？

**6.** 权益 500 万元，每笔风险 1%，止损 40 元/吨，乘数 10，最多几手？

**7.** 毛 Sharpe 1.5、日换手 80%、单边成本 4bp，成本约吃掉多少年化收益？

**8.** 列出至少 6 个回测陷阱，并指出两个在本章代码里被“制造”出来的。

<details><summary>参考答案</summary>

**1.** 统一量纲和波动，防止数值大的信号主导仓位；滚动窗口只用历史数据。全样本 z-score 用了未来均值和方差，是未来函数。

**2.** IC 估计有误差，信号可能高度相关；按 IC 加权会过度集中到“样本内最好”的信号，样本外失效。应用 ICIR、滚动窗口，或先正交化。

**3.** 滞回解决阈值附近抖动，最短停留解决状态频繁切换；代价是反应变慢、转折确认更晚。

**4.** 胜率通常下降，交易次数上升，期望不确定：平均亏损变小，但成本和被扫掉的好单增加。

**5.** $f^*=(0.55\times2.2-1)/1.2=(1.21-1)/1.2=0.175$；半凯利约 0.0875。

**6.** $5000000\times1\%/(40\times10)=125$ 手。

**7.** 双边约 $2\times4\text{bp}\times0.8=6.4\text{bp}=0.064\%$/天，一年约 16%，足以吃掉大部分超额收益。

**8.** 未来函数、幸存者偏差、状态机样本内调参、止损过拟合、忽略成本/冲击、忽略时滞、多重检验、容量幻觉。代码里：信号合成用全样本 IC 估权重；状态机阈值/最短停留、止损倍数都是看着结果挑的。

</details>

---

## 参考资料

- Grinold & Kahn. *Active Portfolio Management*。
- Chan, E. P. *Algorithmic Trading*。
- López de Prado, M. *Advances in Financial Machine Learning*（多重检验、过拟合）。
- Kelly, J. L. (1956). *A New Interpretation of Information Rate*。
- 第07、20、22、35、39章。
